# 04a — PairRisk lesion audit and locked split

This notebook uses only Stage 5 `risk_dev_predictions`. It must never receive a final-test path.

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
PACKAGE_ROOT = REPO_ROOT / "ml_training" / "multiview_conformal"
if not PACKAGE_ROOT.exists():
    PACKAGE_ROOT = Path.cwd().resolve()
if str(PACKAGE_ROOT.parent) not in sys.path:
    sys.path.insert(0, str(PACKAGE_ROOT.parent))

PREDICTIONS = Path("artifacts/rescue/risk_dev/risk_dev_predictions.csv")
IMAGE_ROOT = None  # Set to ISIC_2019_Training_Input when available.
OUTPUT_DIR = Path("artifacts/rescue/pairrisk/design_stage")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(PREDICTIONS, OUTPUT_DIR)

In [ ]:
from multiview_conformal.lesion_bags import (
    load_predictions, collapse_exact_duplicates, build_lesion_table,
    split_design_calibration
)

frame = load_predictions(PREDICTIONS)
deduped, duplicate_audit, duplicate_rows = collapse_exact_duplicates(
    frame, image_root=IMAGE_ROOT
)
lesions = build_lesion_table(deduped)
split = split_design_calibration(lesions)

duplicate_rows.to_csv(OUTPUT_DIR / "duplicate_rows.csv", index=False)
split.to_csv(OUTPUT_DIR / "lesion_split.csv", index=False)
duplicate_audit, split.groupby(["analysis_split", "multi_view"]).size()

**Stop here if image-byte hashing did not run.** Mount the ISIC image directory and rerun before freezing the duplicate audit.